<a href="https://colab.research.google.com/github/maqueda-09/trabajos4to/blob/main/RNN_inflacion_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

RNN - Inflacion


In [3]:
import pandas as pd

# Cargo los datos desde el archivo Excel.
df = pd.read_excel('/content/Consulta_20241113-093525023.xlsx')
# Muestro las primeras 20 filas para ver cómo vienen los datos.
df.head(20)

#  Limpieza Inicial del DataFrame

# El archivo Excel tiene encabezados "sucios" o en filas que no son la primera.
# Selecciono la fila 9 (índice 8) para usarla como nuevo encabezado.
new_header = df.iloc[8]
new_header
# El data frame empieza a ser útil desde la fila 18 (índice 17), así que corto las primeras filas.
df = df[17:]
df
# Asigno la fila que seleccioné antes como los nombres de las columnas.
df.columns = new_header
df.head()
# Reinicio el índice del DataFrame para que empiece en 0 otra vez y elimino el índice viejo.
df.reset_index(drop=True, inplace=True)
df.head()

#  Preprocesamiento de Fechas y Filtrado

# Renombro la columna que tiene el mes/año a 'Fecha' y la convierto a tipo 'datetime'.
# Esto es esencial para trabajar con series de tiempo.
df['Fecha'] = pd.to_datetime(df['Título'])
df.head()
# Muestro la información de las columnas para verificar que los tipos de datos estén correctos.
df.info()

# Filtro el DataFrame para el rango de fechas que quiero analizar.
# Aquí estoy seleccionando datos desde 2014 hasta 2024.
_df = df[(df['Fecha'] >= '01-01-2014') & (df['Fecha'] <= '12-31-2024')]
_df.head()
_df.tail()

# Visualización de los Datos de Inflación

import matplotlib.pyplot as plt

# Obtengo las fechas para el eje X del gráfico.
dates = _df["Fecha"].tolist()

# Grafico la "Índice Nacional de Precios al consumidor, variación anual".
plt.figure(figsize=(16, 10))
plt.plot(dates, _df["Índice Nacional de Precios al consumidor, variación anual"], color='blue', label="Variación anual")
plt.title("Índice Nacional de Precios al Consumidor - variación anual")
plt.xlabel("Tiempo")
plt.ylabel("Variación anual")
plt.xticks(rotation=90) # Roto las fechas para que se lean bien.
plt.legend()
plt.show()


#  Preparación de Datos para el Modelo RNN

# X son las características (los datos de entrada) y 'y' es la variable a predecir.
# Quito la columna de inflación ('y') y las de texto ('Título', 'Fecha') de mi conjunto X.
X = _df.drop(columns=["Índice Nacional de Precios al consumidor, variación anual", "Título", "Fecha"])
# La 'y' es la columna de inflación que quiero predecir.
y = _df["Índice Nacional de Precios al consumidor, variación anual"].values
X.shape
y.shape

# **Escalado de Datos**: Lo hago con MinMaxScaler para que todos los valores estén entre 0 y 1.
# Esto hace que el entrenamiento de la red neuronal sea más estable y rápido.
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 1))
# Escalamos X (las características).
X_scaled = scaler.fit_transform(X)
# Escalamos y (la variable a predecir). Hago un 'reshape' para que sea una matriz de 1 columna.
y_scaled = scaler.fit_transform(y.reshape(-1, 1))

# Creación de Secuencias Temporales (Ventanas)

import numpy as np
# Defino el tamaño de la ventana de tiempo. Uso 12 (probablemente 12 meses) para predecir el siguiente mes.
window_size = 12
X_rnn = []
y_rnn = []
# Construyo las secuencias: tomo 12 pasos de tiempo (meses) para predecir el valor del siguiente.
for i in range(window_size, len(X_scaled)):
    # Agrego al X la ventana de 12 meses anteriores.
    X_rnn.append(X_scaled[i - window_size:i])
    # Agrego a la 'y' el valor de inflación del mes siguiente.
    y_rnn.append(y_scaled[i])

# Convierto las listas a arrays de NumPy, que es lo que espera TensorFlow.
X_rnn, y_rnn = np.array(X_rnn), np.array(y_rnn)

# Divido los datos en entrenamiento y prueba (80% para entrenar).
split = int(len(X) * 0.8)
X_train, y_train = X_rnn[:split], y_rnn[:split]
X_test, y_test = X_rnn[split:], y_rnn[split:]
X_train.shape
y_test.shape
window_size
X_train.shape[2] # Número de características por mes (aparte de la fecha).

# Construcción del Modelo SimpleRNN

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, SimpleRNN, GRU

# Creo el modelo secuencial.
model = Sequential([
  # Primera capa SimpleRNN. Necesito 'return_sequences=True' porque le sigue otra capa RNN.
  # El 'input_shape' es (window_size, número de características).
  SimpleRNN(units=60, return_sequences=True, input_shape=(window_size, X_train.shape[2])),
  # Segunda capa SimpleRNN. Sigue devolviendo secuencias.
  SimpleRNN(units=30, return_sequences=True),
  # Tercera capa SimpleRNN. 'return_sequences=False' porque después viene la capa de salida.
  SimpleRNN(units=15, return_sequences=False),
  # Capa de salida 'Dense' con 1 neurona para la predicción de la inflación.
  Dense(units=1)
])

# Compilación y Entrenamiento

from tensorflow.keras.optimizers import Adam

# Defino el 'learning rate' para el optimizador Adam.
learning_rate = 0.001
adam_optimizer = Adam(learning_rate=learning_rate)
# Compilo el modelo. Uso 'mean_squared_error' (MSE) para medir la pérdida.
model.compile(optimizer=adam_optimizer, loss='mean_squared_error')

# Entreno el modelo con 10 épocas. Uso 'validation_data' para ver qué tan bien se comporta con los datos de prueba durante el entrenamiento.
model.fit(X_train, y_train, batch_size=1, epochs=10, validation_data=(X_test, y_test))

# Evaluación del Modelo

# Hago las predicciones con los datos de prueba.
predictions = model.predict(X_test)
# **Desescalo** las predicciones a los valores originales (porcentajes de inflación).
predictions = scaler.inverse_transform(predictions)
# Desescalo los valores de prueba reales para poder compararlos.
y_test_original = scaler.inverse_transform(y_test)

# Muestro los primeros 5 valores predichos y reales.
print("Predicciones:", predictions.flatten()[:5])
print("Valores reales:", y_test_original.flatten()[:5])

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Calculo las métricas de error.
mae = mean_absolute_error(y_test_original, predictions) # Uso los valores desescalados.
print(f"Error Absoluto Medio (MAE): {mae}")

mse = mean_squared_error(y_test_original, predictions) # Uso los valores desescalados.
print(f"Error Cuadrático Medio (MSE): {mse}")

rmse = np.sqrt(mse)
print(f"Raíz del Error Cuadrático Medio (RMSE): {rmse}")

# Coeficiente de Determinación (R²). Mide qué tan bien se ajusta el modelo a los datos (cercano a 1 es mejor).
r2 = r2_score(y_test_original, predictions) # Uso los valores desescalados.
print(f"Coeficiente de Determinación (R²): {r2}")

#  Gráfico de Resultados

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Tomo la parte del DataFrame que se usó para las pruebas.
valid = _df[split:]
valid = valid.reset_index(drop=True)
valid['Predictions'] = np.nan
# Agrego las predicciones al DataFrame 'valid'. Empiezo después del 'window_size'.
valid.loc[window_size:, 'Predictions'] = predictions

# Preparo las fechas.
dates_valid = pd.to_datetime(valid['Fecha']).apply(lambda x: x.strftime('%Y-%m-%d')).tolist()

# Grafico los valores reales vs. las predicciones.
plt.figure(figsize=(16, 8))
plt.title('Modelo RNN para Predicción de la inflación')
plt.xlabel('Fecha')
plt.ylabel('Inflacion')
plt.plot(dates_valid, valid[["Índice Nacional de Precios al consumidor, variación anual", 'Predictions']])
plt.legend(['Valor Real', 'Predicciones'], loc='lower right')
plt.xticks(rotation=90)
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/content/Consulta_20241113-093525023.xlsx'